<a href="https://colab.research.google.com/github/immaximo/Hybrid-ML-LLM-Tutoring-Model-for-Cloud-Certification-Preparation-/blob/main/SaKT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
!pip install torch torchvision

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install torch_geometric

In [ ]:
import pandas as pd
import numpy as np
import torch

from torch_geometric.data import Data

# =========================
# STEP 1: LOAD DATA
# =========================
interactions = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/synchronized_interaction_logs.csv")          # user_id, question_id, timestamp, correct
questions = pd.read_excel("/content/drive/MyDrive/Colab Notebooks/CDL_150_Questions_With_ID.xlsx")      # question_id, skill_id
googlequestions = pd.read_json("/content/drive/MyDrive/Colab Notebooks/google_cloud_quiz_eval.json")      # question_id, skill_id
studentquiz = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/student_quiz_dataset.csv")      # user_id, skill_id, pre_score, post_score

# Rename 'Question_ID' to 'question_id' in the questions DataFrame to match interactions
questions = questions.rename(columns={'Question_ID': 'question_id'})

# Assign 'Module' as 'skill_id' in the questions DataFrame, assuming Module represents the skill.
# If 'skill_id' is located in a different column, please adjust this line accordingly.
questions['skill_id'] = questions['Module']

# Rename 'student_id' to 'user_id' in interactions DataFrame
interactions = interactions.rename(columns={'student_id': 'user_id'})

# Rename 'student_id' to 'user_id' in studentquiz for merging
studentquiz = studentquiz.rename(columns={'student_id': 'user_id'})

# Merge interactions with studentquiz to get the 'status' (correctness)
# Use a left merge to keep all interactions and add status if available
interactions = interactions.merge(studentquiz[['user_id', 'question_id', 'status']],
                                  on=['user_id', 'question_id'],
                                  how='left')

# Convert 'status' to numerical 'correct' column (1 for 'correct', 0 for 'incorrect')
# Fill NaN values (for interactions without a matching status) with 0, marking them as incorrect by default
interactions['correct'] = interactions['status'].map({'correct': 1, 'incorrect': 0}).fillna(0).astype(int)

# Drop the original 'status' column as it's no longer needed
interactions = interactions.drop(columns=['status'], errors='ignore')

# =========================
# STEP 2: MERGE INTERACTIONS WITH SKILLS
# =========================
data = interactions.merge(questions, on="question_id", how="inner")
data = data.sort_values(by=["user_id", "timestamp"])

# Encode IDs
user2idx = {u: i for i, u in enumerate(data["user_id"].unique())}
q2idx = {q: i for i, q in enumerate(data["question_id"].unique())}
s2idx = {s: i for i, s in enumerate(data["skill_id"].unique())}

data["user_id"] = data["user_id"].map(user2idx)
data["question_id"] = data["question_id"].map(q2idx)
data["skill_id"] = data["skill_id"].map(s2idx)

num_users = len(user2idx)
num_skills = len(s2idx)

# =========================
# STEP 3: BUILD STUDENT SEQUENCES
# =========================
student_sequences = []
grouped = data.groupby("user_id")

for user, group in grouped:
    student_sequences.append({
        "student_id": user,
        "q_seq": torch.tensor(group["question_id"].values, dtype=torch.long),
        "s_seq": torch.tensor(group["skill_id"].values, dtype=torch.long),
        "r_seq": torch.tensor(group["correct"].values, dtype=torch.float)
    })

# =========================
# STEP 4: BUILD SKILL GRAPH
# =========================
adj_matrix = np.zeros((num_skills, num_skills))
for seq in student_sequences:
    skills = seq["s_seq"].numpy()
    for i in range(len(skills)-1):
        s1, s2 = skills[i], skills[i+1]
        adj_matrix[s1, s2] += 1
        adj_matrix[s2, s1] += 1

# Convert to PyG edge_index
edges = np.array(np.nonzero(adj_matrix))
edge_index = torch.tensor(edges, dtype=torch.long)
edge_weight = torch.tensor(adj_matrix[edges[0], edges[1]], dtype=torch.float)

# Node features: one-hot for skills
x = torch.eye(num_skills, dtype=torch.float)

# =========================
# STEP 5: CREATE PyG DATA OBJECT
# =========================
data_gnn = Data(x=x, edge_index=edge_index, edge_attr=edge_weight)
data_gnn.student_sequences = student_sequences

# =========================
# STEP 6: SAVE DATA
# =========================
torch.save(data_gnn, "kt_all_models_prepost.pt")
print("Preprocessing complete. Dataset ready for all models (AKT, DKT, BKT, GRKT/GNN).")

In [ ]:
import torch.nn as nn
from torch_geometric.nn import GCNConv # This import is not used by SAKT, but might be used by GRKT if it were kept.
import requests

# Fetch SAKT model and its dependencies from GitHub
sakt_url = "https://github.com/pykt-team/pykt-toolkit/raw/main/pykt/models/sakt.py"
utils_url = "https://github.com/pykt-team/pykt-toolkit/raw/main/pykt/models/utils.py"

response_sakt = requests.get(sakt_url)
response_sakt.raise_for_status()

response_utils = requests.get(utils_url)
response_utils.raise_for_status()

with open("sakt.py", "w") as f:
    sakt_content = response_sakt.text
    # Replace relative import with absolute import for local file
    sakt_content = sakt_content.replace("from .utils import", "from utils import")
    f.write(sakt_content)

with open("utils.py", "w") as f:
    f.write(response_utils.text)

from sakt import SAKT # Import SAKT

In [ ]:
!pip install optuna

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, mean_squared_error
import numpy as np
import optuna
from sakt import SAKT

# Load the preprocessed data
data_gnn = torch.load("kt_all_models_prepost.pt", weights_only=False)
student_sequences = data_gnn.student_sequences

# Get the number of skills and questions from the preprocessed data
num_skills = data_gnn.x.shape[0]
num_questions = max([max(seq['q_seq']).item() for seq in student_sequences]) + 1
max_seq_len = max([len(seq['q_seq']) for seq in student_sequences])

# Padding index for skill sequences and question sequences
padding_idx_s = num_skills
padding_idx_q = num_questions

# --- Custom Dataset and Collate Function for DataLoader ---
class SAKTDataset(Dataset):
    def __init__(self, sequences, max_seq_len):
        self.sequences = sequences
        self.max_seq_len = max_seq_len

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        seq = self.sequences[idx]
        return seq['q_seq'], seq['s_seq'], seq['r_seq']

def collate_fn(batch):
    q_seqs = [item[0] for item in batch]
    s_seqs = [item[1] for item in batch]
    r_seqs = [item[2] for item in batch]

    # Pad sequences to the max length in the current batch
    max_len = max([len(s) for s in s_seqs])

    padded_q_seqs = torch.stack([
        torch.cat([q, torch.full((max_len - len(q),), padding_idx_q, dtype=torch.long)])
        for q in q_seqs
    ])
    padded_s_seqs = torch.stack([
        torch.cat([s, torch.full((max_len - len(s),), padding_idx_s, dtype=torch.long)])
        for s in s_seqs
    ])
    padded_r_seqs = torch.stack([
        torch.cat([r, torch.full((max_len - len(r),), -1.0, dtype=torch.float)])
        for r in r_seqs
    ])

    # Create a mask to ignore padded values in loss calculation
    mask = (padded_r_seqs != -1.0)

    return padded_q_seqs, padded_s_seqs, padded_r_seqs, mask

# --- Data Splitting (Train/Validation/Test) ---
train_size = int(0.8 * len(student_sequences))
val_size = int(0.1 * len(student_sequences))
test_size = len(student_sequences) - train_size - val_size

# Ensure reproducibility of the split
torch.manual_seed(42)
train_sequences, val_sequences, test_sequences = torch.utils.data.random_split(
    student_sequences, [train_size, val_size, test_size]
)

train_sakt_dataset = SAKTDataset(train_sequences, max_seq_len)
val_sakt_dataset = SAKTDataset(val_sequences, max_seq_len)
test_sakt_dataset = SAKTDataset(test_sequences, max_seq_len)

# --- Device Configuration ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# --- SAM Optimizer Implementation ---
class SAM(torch.optim.Optimizer):
    def __init__(self, params, base_optimizer, rho=0.05, adaptive=False, **kwargs):
        assert rho >= 0.0, f"Invalid rho, should be non-negative: {rho}"
        defaults = dict(rho=rho, adaptive=adaptive, **kwargs)
        super(SAM, self).__init__(params, defaults)
        self.base_optimizer = base_optimizer(self.param_groups, **kwargs)
        self.param_groups = self.base_optimizer.param_groups

    @torch.no_grad()
    def first_step(self, zero_grad=False):
        grad_norm = self._grad_norm()
        for group in self.param_groups:
            scale = group["rho"] / (grad_norm + 1e-12)
            for p in group["params"]:
                if p.grad is None: continue
                self.state[p]["old_p"] = p.data.clone()
                e_w = (torch.pow(p, 2) if group["adaptive"] else 1.0) * p.grad * scale.to(p)
                p.add_(e_w)
        if zero_grad: self.zero_grad()

    @torch.no_grad()
    def second_step(self, zero_grad=False):
        for group in self.param_groups:
            for p in group["params"]:
                if p.grad is None: continue
                p.data = self.state[p]["old_p"]
        self.base_optimizer.step()
        if zero_grad: self.zero_grad()

    @torch.no_grad()
    def step(self, closure=None):
        assert closure is not None, "Sharpness Aware Minimization requires closure"
        closure = torch.enable_grad()(closure)
        self.first_step(zero_grad=True)
        closure()
        self.second_step()

    def _grad_norm(self):
        shared_device = self.param_groups[0]["params"][0].device
        norm = torch.norm(
                    torch.stack([
                        ((torch.abs(p) if group["adaptive"] else 1.0) * p.grad).norm(p=2).to(shared_device)
                        for group in self.param_groups for p in group["params"]
                        if p.grad is not None
                    ]),
                    p=2
               )
        return norm

# --- Evaluation Function ---
def evaluate_model(model, dataloader, device):
    model.eval()
    all_predictions = []
    all_targets = []
    with torch.no_grad():
        for q_seqs, s_seqs, r_seqs, mask in dataloader:
            q_seqs, s_seqs, r_seqs, mask = q_seqs.to(device), s_seqs.to(device), r_seqs.to(device), mask.to(device)
            r_seqs_safe = torch.where(r_seqs == -1.0, torch.zeros_like(r_seqs), r_seqs).long()

            out = model(q_seqs, r_seqs_safe, q_seqs)
            predictions = out[0] if isinstance(out, tuple) else out

            if predictions.dim() > 2:
                predictions = predictions.squeeze(-1)

            masked_predictions = predictions[mask]
            masked_targets = r_seqs[mask]

            all_predictions.extend(masked_predictions.cpu().numpy())
            all_targets.extend(masked_targets.cpu().numpy())

    if len(all_targets) == 0:
        return 0.0, 0.0

    auc = roc_auc_score(all_targets, all_predictions)
    rmse = np.sqrt(mean_squared_error(all_targets, all_predictions))
    return auc, rmse

# --- Optuna Objective Function ---
def objective(trial):
    # Balanced capacity to reach 0.7-0.8 over 10 epochs (increased LR slightly)
    hidden_dim = trial.suggest_categorical("hidden_dim", [32, 64])
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 8e-4, log=True)
    dropout_rate = trial.suggest_float("dropout_rate", 0.3, 0.5)
    rho = trial.suggest_float("rho", 0.01, 0.05)
    batch_size = trial.suggest_categorical("batch_size", [128, 256])
    num_heads = trial.suggest_categorical("num_heads", [2])

    train_loader = DataLoader(train_sakt_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
    val_loader = DataLoader(val_sakt_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

    # Initialize SAKT Model
    model = SAKT(num_c=num_questions + 1, seq_len=max_seq_len, emb_size=hidden_dim, num_attn_heads=num_heads, dropout=dropout_rate)
    model.to(device)

    base_optimizer = torch.optim.Adam
    optimizer = SAM(model.parameters(), base_optimizer, rho=rho, lr=learning_rate)
    criterion = nn.BCELoss()

    epochs = 5 # Short tuning phase

    for epoch in range(epochs):
        model.train()
        for q_seqs, s_seqs, r_seqs, mask in train_loader:
            q_seqs, s_seqs, r_seqs, mask = q_seqs.to(device), s_seqs.to(device), r_seqs.to(device), mask.to(device)
            r_seqs_safe = torch.where(r_seqs == -1.0, torch.zeros_like(r_seqs), r_seqs).long()

            def closure():
                optimizer.zero_grad()
                out = model(q_seqs, r_seqs_safe, q_seqs)
                predictions = out[0] if isinstance(out, tuple) else out

                if predictions.dim() > 2:
                    predictions = predictions.squeeze(-1)

                masked_predictions = predictions[mask]
                masked_targets = r_seqs[mask]
                if len(masked_targets) == 0:
                     return torch.tensor(0.0, requires_grad=True).to(device)
                loss = criterion(masked_predictions, masked_targets)
                loss.backward()
                return loss

            if len(r_seqs[mask]) > 0:
                 closure()
                 optimizer.step(closure)

    val_auc, _ = evaluate_model(model, val_loader, device)
    return val_auc

print("Starting Hyperparameter Tuning with Optuna...")
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=3) # Reduced to 3 trials

print("Best trial:")
trial = study.best_trial
print(f"  Value (AUC): {trial.value}")

# --- Retrain Best Model and Evaluate on Test Set ---
print("\n--- Retraining Best Model ---")
best_hidden_dim = trial.params["hidden_dim"]
best_lr = trial.params["learning_rate"]
best_dropout = trial.params["dropout_rate"]
best_rho = trial.params["rho"]
best_batch_size = trial.params["batch_size"]
best_num_heads = trial.params["num_heads"]

best_train_loader = DataLoader(train_sakt_dataset, batch_size=best_batch_size, shuffle=True, collate_fn=collate_fn)
best_val_loader = DataLoader(val_sakt_dataset, batch_size=best_batch_size, shuffle=False, collate_fn=collate_fn)
best_test_loader = DataLoader(test_sakt_dataset, batch_size=best_batch_size, shuffle=False, collate_fn=collate_fn)

best_model = SAKT(num_c=num_questions + 1, seq_len=max_seq_len, emb_size=best_hidden_dim, num_attn_heads=best_num_heads, dropout=best_dropout)
best_model.to(device)

base_optimizer = torch.optim.Adam
best_optimizer = SAM(best_model.parameters(), base_optimizer, rho=best_rho, lr=best_lr)
criterion = nn.BCELoss()

# Target exactly 10 epochs
epochs = 10
best_val_auc = -1.0

for epoch in range(epochs):
    best_model.train()
    for q_seqs, s_seqs, r_seqs, mask in best_train_loader:
        q_seqs, s_seqs, r_seqs, mask = q_seqs.to(device), s_seqs.to(device), r_seqs.to(device), mask.to(device)
        r_seqs_safe = torch.where(r_seqs == -1.0, torch.zeros_like(r_seqs), r_seqs).long()

        def closure():
            best_optimizer.zero_grad()
            out = best_model(q_seqs, r_seqs_safe, q_seqs)
            predictions = out[0] if isinstance(out, tuple) else out

            if predictions.dim() > 2:
                predictions = predictions.squeeze(-1)

            masked_predictions = predictions[mask]
            masked_targets = r_seqs[mask]
            if len(masked_targets) == 0:
                 return torch.tensor(0.0, requires_grad=True).to(device)
            loss = criterion(masked_predictions, masked_targets)
            loss.backward()
            return loss

        if len(r_seqs[mask]) > 0:
             closure()
             best_optimizer.step(closure)

    val_auc, val_rmse = evaluate_model(best_model, best_val_loader, device)
    print(f"Epoch {epoch+1}/{epochs}: Val AUC: {val_auc:.4f}, Val RMSE: {val_rmse:.4f}")

    if val_auc > best_val_auc:
        best_val_auc = val_auc
        torch.save(best_model.state_dict(), "best_sakt_model_tuned.pt")

print("\n--- Evaluating on Test Set ---")
best_model.load_state_dict(torch.load("best_sakt_model_tuned.pt"))
test_auc, test_rmse = evaluate_model(best_model, best_test_loader, device)
print(f"Test AUC: {test_auc:.4f}, Test RMSE: {test_rmse:.4f}")


In [ ]:
import time
import torch
import random

print("Calculating detailed latency and predictions per question for SAKT...")

best_model.eval() # Ensure SAKT model is in evaluation mode
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
best_model.to(device)

# Choose 3 random original question IDs from the available mapping
all_original_q_ids = list(q2idx.keys())
target_original_q_ids = random.sample(all_original_q_ids, 3)

results = []

with torch.no_grad():
    for original_q_id in target_original_q_ids:
        # 1. Get encoded question ID
        encoded_q_id = q2idx[original_q_id]

        # 2. Get original skill ID from the 'questions' DataFrame
        # Assuming 'question_id' in 'questions' df refers to the original_q_id
        original_skill_id = questions[questions['question_id'] == original_q_id]['skill_id'].iloc[0]

        # 3. Get encoded skill ID
        encoded_skill_id = s2idx[original_skill_id]

        # Create dummy input for a single prediction for SAKT
        # SAKT predicts the outcome of the target question given the sequence.
        # We provide a sequence of length 1 for consistency.
        input_q_tensor = torch.tensor([[encoded_q_id]], dtype=torch.long).to(device)
        input_r_tensor = torch.tensor([[0]], dtype=torch.long).to(device) # Dummy response (e.g., incorrect)

        start_time = time.perf_counter() # Start timing for this specific prediction

        # SAKT model forward pass: model(q_seq, r_seq, target_q)
        out = best_model(input_q_tensor, input_r_tensor, input_q_tensor)
        preds = out[0] if isinstance(out, tuple) else out
        if preds.dim() > 2:
            preds = preds.squeeze(-1)

        # Get the prediction (for the only item in our sequence)
        predicted_probability = preds[0, -1].item()

        end_time = time.perf_counter() # End timing

        latency_ms = (end_time - start_time) * 1000 # Convert to milliseconds

        results.append({
            "Question ID": original_q_id,
            "Encoded Skill ID": encoded_skill_id,
            "Predicted Probability (Correct)": predicted_probability,
            "Latency": latency_ms
        })

print("Target Questions and their Encoded Skill IDs:")
for original_q_id in target_original_q_ids:
    if original_q_id in q2idx:
        # Get original skill ID and then encoded skill ID
        original_skill_id = questions[questions['question_id'] == original_q_id]['skill_id'].iloc[0]
        encoded_skill_id = s2idx[original_skill_id]
        print(f"  {{'{original_q_id}': {encoded_skill_id}}}")

print("\nLatency Test Results:")
for res in results:
    print(f"Question ID: {res['Question ID']}")
    print(f"  Encoded Skill ID: {res['Encoded Skill ID']}")
    print(f"  Predicted Probability (Correct): {res['Predicted Probability (Correct)']:.4f}")
    print(f"  Latency: {res['Latency']:.4f} ms")

print("\nDetailed latency calculation complete.")
